# Ejercicio 9: Uso de una API de LLM (Groq Cloud)

Daniel Flores

## Panorama general

Una API de LLM es la puerta de entrada a un modelo de lenguaje grande. En vez de descargarlo y correrlo en mi máquina (que requiere GPU y mucha RAM), mando una petición HTTP con mi pregunta y el servidor me devuelve la respuesta. Es lo mismo que hace ChatGPT por debajo cuando uno escribe en la web.

El cuaderno original menciona Google Gemini, pero acá voy a usar **Groq Cloud**. Groq corre modelos open source (Llama, Gemma, Mixtral) sobre chips propios llamados LPUs (Language Processing Units). El detalle clave: es rapidísimo. Donde otros tardan 5 segundos, Groq devuelve la respuesta en menos de 1. Tiene plan free con cuota diaria de tokens y requests por minuto.

### ¿Para qué sirve esto en la vida real?
Casi todo lo que vemos hoy con IA generativa se construye así. Un chatbot de atención al cliente, un asistente que resume contratos, un buscador semántico con respuestas en lenguaje natural (RAG), un agente que ejecuta tareas. Todo eso es código que llama a una API de LLM y le pasa un prompt bien armado.


## 1. Uso básico

Primero lo más simple: mandar un mensaje y recibir la respuesta.

In [ ]:
!pip install openai scikit-learn sentence-transformers numpy pandas python-dotenv tenacity --quiet

In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

# Lee las variables del archivo .env (que está en .gitignore, no se sube al repo)
load_dotenv()

# Groq Cloud usa el mismo SDK que OpenAI. Solo cambia la URL base y la API key.
client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ["GROQ_API_KEY"]
)

# Modelo del plan free de Groq
MODELO = "llama-3.3-70b-versatile"

### Explicación línea por línea

- `import os`: para leer variables de entorno del sistema.
- `from dotenv import load_dotenv`: librería que carga un archivo `.env` y mete sus variables en `os.environ`.
- `load_dotenv()`: lee el `.env` que está al lado del notebook. Adentro tengo `GROQ_API_KEY=gsk_...`.
- `client = OpenAI(...)`: creo el cliente. Es un objeto que sabe cómo hablar con el servidor.
- `base_url="https://api.groq.com/openai/v1"`: le digo a dónde enviar las peticiones. Groq implementa el mismo protocolo que OpenAI.
- `api_key=os.environ["GROQ_API_KEY"]`: leo la clave desde la variable de entorno. Así la clave no queda escrita en el código y no se sube a GitHub.
- `MODELO = "llama-3.3-70b-versatile"`: el Llama 3.3 de 70B servido por Groq, gratis dentro de la cuota diaria.

*Buena práctica:* nunca pongas claves API en el código. GitHub tiene un escáner que bloquea pushes con secretos visibles, y aunque pase, el repo puede quedar público y cualquiera la usa.

In [20]:
# pip install tenacity
from tenacity import retry, wait_exponential, stop_after_attempt, retry_if_exception_type
from openai import RateLimitError

# min=30 obliga a esperar al menos 30s, superando los 29s del Rate Limit
@retry(
    wait=wait_exponential(multiplier=1, min=30, max=120), 
    stop=stop_after_attempt(4),
    retry=retry_if_exception_type(RateLimitError)
)
def llamar_api():
    return client.chat.completions.create(
        model=MODELO,
        messages=[
            {"role": "user", "content": "Hola, dime en una frase qué es la recuperación de información."}
        ]
    )

respuesta = llamar_api()
print(respuesta.choices[0].message.content)

La recuperación de información es el proceso de ubicar y recuperar datos o documentos relevantes de una gran cantidad de información almacenada, utilizando técnicas y algoritmos para encontrar y retrieval de la información solicitada de manera eficiente.


### Cómo se lee este bloque

- `client.chat.completions.create(...)`: la llamada principal. Crea un *chat completion* (una respuesta tipo chat).
- `model=MODELO`: qué modelo va a responder.
- `messages=[...]`: la conversación. Es una lista de diccionarios con dos llaves: `role` y `content`. El `role` puede ser `user` (yo), `assistant` (la IA en turnos anteriores) o `system` (instrucciones globales).
- `respuesta.choices[0].message.content`: la API devuelve un objeto. `choices` es una lista porque podría haber varias respuestas alternativas. Tomo la primera, su `message`, y dentro su `content` (el texto).

*Dato curioso de estructuras de datos:* el historial del chat es una lista enlazada por orden. Cada turno se *appendea*. Es la misma idea de una cola FIFO si quisiera limitar la memoria del chat.

## 2. Retrieval

Ahora la parte interesante. Voy a hacer un mini buscador semántico sobre el corpus **20 News Groups**, que son ~18 mil posts agrupados en 20 temas (autos, política, religión, computadoras, etc.). Es un dataset clásico de procesamiento de lenguaje natural.

El plan: cargo los textos, los convierto en vectores con un modelo de embeddings, recibo una *query*, la convierto también en vector y devuelvo los 5 documentos más parecidos por similitud coseno.


### 2.1 Cargo el corpus de 20 News Groups

In [4]:
from sklearn.datasets import fetch_20newsgroups

# Descargo el corpus. remove=('headers','footers','quotes') quita el ruido típico de email.
datos = fetch_20newsgroups(
    subset="train",
    remove=("headers", "footers", "quotes")
)

documentos = datos.data
etiquetas = datos.target_names

print("Total documentos:", len(documentos))
print("Categorías:", etiquetas)
print("\nEjemplo:\n", documentos[0][:300])

Total documentos: 11314
Categorías: ['alt.atheism', 'comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware', 'comp.windows.x', 'misc.forsale', 'rec.autos', 'rec.motorcycles', 'rec.sport.baseball', 'rec.sport.hockey', 'sci.crypt', 'sci.electronics', 'sci.med', 'sci.space', 'soc.religion.christian', 'talk.politics.guns', 'talk.politics.mideast', 'talk.politics.misc', 'talk.religion.misc']

Ejemplo:
 I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I k


### Lectura del código

- `fetch_20newsgroups`: función de scikit-learn que descarga el corpus la primera vez y luego lo cachea.
- `subset="train"`: hay split de entrenamiento y test. Uso solo train porque me basta.
- `remove=("headers","footers","quotes")`: cada post original trae cabeceras de email y citas de mensajes anteriores. Eso ensucia mucho los embeddings, así que lo quito.
- `datos.data`: la lista de textos.
- `datos.target_names`: los nombres de las 20 categorías.

Son ~11 mil documentos. Generar embeddings para todos en CPU tarda bastante, así que voy a quedarme con una muestra de 2000 para que el cuaderno corra rápido.

In [5]:
import numpy as np

np.random.seed(42)  # reproducibilidad: misma muestra siempre

N = 2000
indices = np.random.choice(len(documentos), size=N, replace=False)
corpus = [documentos[i] for i in indices]

print("Corpus reducido a:", len(corpus), "documentos")

Corpus reducido a: 2000 documentos


- `np.random.seed(42)`: fijo la semilla. Sin esto cada corrida tendría una muestra distinta y los resultados no se podrían comparar.
- `np.random.choice(...)`: elige N índices al azar sin repetir.
- La *list comprehension* arma la lista nueva con esos índices.

### 2.2 Transformo a embeddings

Un *embedding* es un vector que representa el significado de un texto. Dos textos parecidos en sentido caen cerca en ese espacio. Voy a usar `all-MiniLM-L6-v2`, que es chico (384 dimensiones) y rápido en CPU.

In [6]:
from sentence_transformers import SentenceTransformer

modelo_emb = SentenceTransformer("all-MiniLM-L6-v2")

print("Generando embeddings...")
embeddings = modelo_emb.encode(
    corpus,
    batch_size=64,
    show_progress_bar=True
)

print("Shape de la matriz:", embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\DELL\AppData\Roaming\Python\Python312\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\DELL\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generando embeddings...


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Shape de la matriz: (2000, 384)


### Qué hace cada parte

- `SentenceTransformer("all-MiniLM-L6-v2")`: carga el modelo (lo baja de Hugging Face la primera vez).
- `modelo_emb.encode(corpus, ...)`: pasa cada texto por el modelo y devuelve su vector. `batch_size=64` significa que procesa de a 64 textos a la vez para acelerar.
- El resultado es una matriz `(2000, 384)`. Una fila por documento, 384 números por fila.

*Dato curioso de machine learning:* esos 384 números son la salida de la última capa del transformer. El modelo fue entrenado para que oraciones con el mismo significado den vectores parecidos. Es el mismo principio detrás de Word2Vec que vimos antes, pero a nivel de oración.

### 2.3 Creo una query y hago la búsqueda

In [7]:
from sklearn.metrics.pairwise import cosine_similarity

def buscar(query, top_k=5):
    # 1. embedding de la query
    q_vec = modelo_emb.encode([query])
    
    # 2. similitud coseno contra todos los documentos
    sims = cosine_similarity(q_vec, embeddings)[0]
    
    # 3. índices de los top_k más altos
    top = np.argsort(sims)[::-1][:top_k]
    
    # 4. devuelvo lista de tuplas (similitud, texto)
    return [(sims[i], corpus[i]) for i in top]

### Línea por línea

- `modelo_emb.encode([query])`: meto la query en una lista porque `encode` espera lista. Devuelve un vector de shape `(1, 384)`.
- `cosine_similarity(q_vec, embeddings)`: scikit calcula el coseno entre la query y los 2000 documentos. Devuelve matriz `(1, 2000)`. El `[0]` la aplana a array de 2000 números.
- `np.argsort(sims)`: ordena de menor a mayor y devuelve los índices, no los valores.
- `[::-1]`: lo invierte (ahora va de mayor a menor).
- `[:top_k]`: me quedo con los primeros K.
- Y devuelvo la similitud junto al texto para mostrarlo bien.

*Dato curioso de algoritmos:* `argsort` es O(n log n). Para corpus enormes esto se vuelve lento, y ahí entran las bases de datos vectoriales (FAISS, Pinecone) que usan estructuras tipo HNSW para buscar vecinos cercanos en tiempo casi constante.

In [8]:
query = "problems with car engine and motor oil"
resultados = buscar(query, top_k=5)

print(f"Query: {query}\n")
print("="*60)
for i, (sim, texto) in enumerate(resultados, start=1):
    print(f"\n[{i}] Similitud: {sim:.4f}")
    print("-"*60)
    print(texto[:400])

Query: problems with car engine and motor oil


[1] Similitud: 0.4370
------------------------------------------------------------
As an additional data point, I have run Castrol 20W50 exclusively
in the following cars: 75 Rabbit, 78 Scirocco, 76 Rabbit, 78 Bus,
70 Beetle, 76 Bus, 86 Jetta GLI.  I've never had an oil-related
problem.

Disclaimer:  It gets mighty hot down here.

[2] Similitud: 0.4037
------------------------------------------------------------

You can avoid these problems entirely by installing an oil drain valve in
place of the bolt.  I have one on both of my cars.  There have been no
leaks in 210,000 miles (combined miles on both cars).

[3] Similitud: 0.3384
------------------------------------------------------------
]Just wanted to say "Thanks" to everyone who sent me e-mail or
]posted a reply to my question on the oil consumption in my K75S

so what did _you_ decide?


[4] Similitud: 0.2839
------------------------------------------------------------
I have a lin

Acá obtengo los 5 documentos más similares a mi query. Si todo salió bien, deberían aparecer posts del grupo `rec.autos`.

## 3. Experimentación

Hasta acá está lo básico que pide el cuaderno. Ahora voy a meter cosas extra que me parecen interesantes.

### 3.1 Comparar varias queries de distintos temas

Quiero ver si el buscador discrimina bien entre temas muy distintos.

In [9]:
queries = [
    "christianity and the bible",
    "NASA space shuttle launch",
    "how to encrypt my email",
    "hockey playoffs season"
]

for q in queries:
    print("="*60)
    print("Query:", q)
    res = buscar(q, top_k=3)
    for i, (sim, texto) in enumerate(res, 1):
        print(f"  [{i}] sim={sim:.3f} :: {texto[:120].strip()}")
    print()

Query: christianity and the bible
  [1] sim=0.463 :: Just a few cheap shots a Christianity:

Riddle: What is the shortest street in Jerusalem?
Answer: The Street of the Righ
  [2] sim=0.458 :: This is, as far as I know, complete nonsense.  The codification of the bible
as we have it now came very much later.
  [3] sim=0.435 :: /(emery)
/The one single historic event that has had the biggest impact on the
/world over the centuries is the resurrec

Query: NASA space shuttle launch
  [1] sim=0.417 :: Ed Campion
Headquarters, Washington, D.C.                             April 23, 1993
(Phone:  202/358-1780)

Kyle Herrin
  [2] sim=0.399 :: COMET (Commercial Experiment Transport) is to launch from Wallops Island
Virginia and orbit Earth for about 30 days. It
  [3] sim=0.394 :: ------------------------------------------------------------------------------

Ocean Reconnaissance Launch Surprises

Query: how to encrypt my email
  [1] sim=0.464 :: Archive-name: cryptography-faq/part08
Last-modifi

### 3.2 RAG: que el LLM responda usando los documentos recuperados

Esto es lo que se llama **Retrieval-Augmented Generation**. El LLM responde no con lo que sabe de entrenamiento, sino con lo que le paso como contexto. Es lo que usan los chatbots empresariales que responden sobre documentos internos.

In [10]:
def responder_rag(pregunta, top_k=3):
    # recupero contexto
    docs = buscar(pregunta, top_k=top_k)
    contexto = "\n\n---\n\n".join([texto[:500] for _, texto in docs])
    
    # armo el prompt con el contexto inyectado
    system_msg = (
        "Eres un asistente que responde solo con base en el contexto entregado. "
        "Si la respuesta no está en el contexto, dilo claramente.\n\n"
        f"CONTEXTO:\n{contexto}"
    )
    
    resp = client.chat.completions.create(
        model=MODELO,
        messages=[
            {"role": "system", "content": system_msg},
            {"role": "user", "content": pregunta}
        ]
    )
    return resp.choices[0].message.content

pregunta = "What do people say about car engine problems?"
print("Pregunta:", pregunta)
print("\nRespuesta del RAG:\n")
print(responder_rag(pregunta))

Pregunta: What do people say about car engine problems?

Respuesta del RAG:

No hay información en el contexto sobre problemas con los motores de los carros en general. El contexto menciona una posible escasez del motor de 3.5L para los carros LH de Chrysler, pero no se discuten problemas con los motores en sí.


### 3.3 Comparar similitud coseno vs producto punto

Curioso ver si cambia el ranking si uso producto punto puro en vez de coseno. Como `all-MiniLM-L6-v2` ya devuelve vectores normalizados (norma ~1), debería dar lo mismo o muy parecido.

In [11]:
query = "computer graphics and 3D rendering"
q_vec = modelo_emb.encode([query])

sims_cos = cosine_similarity(q_vec, embeddings)[0]
sims_dot = (q_vec @ embeddings.T)[0]  # producto punto

top_cos = set(np.argsort(sims_cos)[::-1][:5])
top_dot = set(np.argsort(sims_dot)[::-1][:5])

print("Top 5 con coseno   :", top_cos)
print("Top 5 con dot      :", top_dot)
print("Coincidencias      :", len(top_cos & top_dot), "/ 5")

Top 5 con coseno   : {1069, 1776, 629, 309, 118}
Top 5 con dot      : {1069, 1776, 629, 309, 118}
Coincidencias      : 5 / 5


### 3.4 Mini chatbot con memoria sobre el corpus

Junto las dos cosas: el RAG y un historial corto. Cada turno recupero contexto nuevo y agrego el historial previo.

In [12]:
from collections import deque

class ChatRAG:
    def __init__(self, max_turnos=3):
        self.historial = deque(maxlen=max_turnos * 2)  # *2 por user+assistant
    
    def hablar(self, pregunta):
        docs = buscar(pregunta, top_k=2)
        contexto = "\n\n".join([t[:400] for _, t in docs])
        
        msgs = [{"role": "system", "content": f"Responde usando este contexto:\n{contexto}"}]
        msgs.extend(self.historial)
        msgs.append({"role": "user", "content": pregunta})
        
        resp = client.chat.completions.create(model=MODELO, messages=msgs)
        contenido = resp.choices[0].message.content
        
        self.historial.append({"role": "user", "content": pregunta})
        self.historial.append({"role": "assistant", "content": contenido})
        return contenido

chat = ChatRAG(max_turnos=3)

print("USER: Tell me about a topic discussed in the corpus related to space.")
print("BOT:", chat.hablar("Tell me about a topic discussed in the corpus related to space."))
print()
print("USER: And what specific missions are mentioned?")
print("BOT:", chat.hablar("And what specific missions are mentioned?"))

USER: Tell me about a topic discussed in the corpus related to space.
BOT: One topic discussed in the corpus related to space is the existence of a forwarding system for sci.space posts called "Space Digest". This system mirrors the sci.space Usenet group and provides two-way communication, allowing users to stay up-to-date with posts and discussions related to space. 

Additionally, the corpus mentions that there is a series of linked messages posted to the sci.space and sci.astro Usenet groups that provide answers to frequently asked questions and reference material related to space and astronomy.

USER: And what specific missions are mentioned?
BOT: There is no specific mention of space missions in the provided corpus. However, it does mention a "Manned Lunar Exploration conference" that is scheduled to take place on May 7th at Crystal City, Virginia, under the auspices of the American Institute of Aeronautics and Astronautics (AIAA). This suggests that the conference may be related

*Dato curioso de estructuras de datos:* la `deque` con `maxlen` es una cola circular. Cuando supera el tamaño, descarta el más viejo automáticamente. Sin esto el historial crecería sin límite y los costos por token se dispararían.

### 3.5 Ver qué tan agrupados quedan los embeddings

Tomo dos queries muy distintas y comparo la similitud promedio de sus top 20. Si los embeddings funcionan, la similitud de la query con su top debería ser bastante mayor que con documentos al azar.

In [13]:
import random
random.seed(0)

query = "religion and faith"
q_vec = modelo_emb.encode([query])
sims = cosine_similarity(q_vec, embeddings)[0]

top_20 = np.sort(sims)[::-1][:20]
azar = sims[random.sample(range(len(sims)), 20)]

print(f"Similitud promedio top 20 : {top_20.mean():.4f}")
print(f"Similitud promedio al azar: {azar.mean():.4f}")
print(f"Diferencia                : {top_20.mean() - azar.mean():.4f}")

Similitud promedio top 20 : 0.3360
Similitud promedio al azar: 0.0243
Diferencia                : 0.3117
